# Codificação de Variáveis Categóricas - Parte 2
## Técnicas Avançadas e Comparações

**Disciplina:** INF1032 - Introdução à Ciência de Dados  
**Professor:** Adriano Branco  
**Instituição:** PUC-Rio - Departamento de Informática  

### Aula 08 - Parte 2

---

### 🎯 **Objetivos desta parte:**
- Aprender técnicas avançadas: Ordinal, Target, Hash e Frequency Encoding
- Compreender os riscos do Target Encoding e como mitigá-los
- Aplicar estratégias para lidar com alta cardinalidade
- Comparar o impacto dimensional das diferentes técnicas
- Desenvolver critérios para escolha da técnica adequada


---

In [ ]:
# @title
# Inicialização e importação de bibliotecas
# Importação das bibliotecas necessárias (caso esteja executando apenas a Parte 2)
!pip install -q category_encoders
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, OrdinalEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from category_encoders import BinaryEncoder, TargetEncoder, HashingEncoder
import warnings
warnings.filterwarnings('ignore')

# Configurações de visualização
plt.style.use('default')
sns.set_palette('viridis')
plt.rcParams['figure.figsize'] = (12, 6)

# Nota: Se você executou a Parte 1, o DataFrame 'df' já está carregado.
# Caso contrário, execute o código de geração de dados abaixo:

# Verificar se df existe, se não, criar os dados
if 'df' not in locals():
    print("Gerando dados sintéticos...")
    np.random.seed(42)
    n = 1000

    df = pd.DataFrame({
        'sexo': np.random.choice(['Masculino', 'Feminino'], n, p=[0.45, 0.55]),
        'estado_civil': np.random.choice(['Solteiro', 'Casado', 'Divorciado', 'Viúvo'], n, p=[0.35, 0.45, 0.15, 0.05]),
        'nivel_educacao': np.random.choice(['Ensino Fundamental', 'Ensino Médio', 'Graduação',
                                           'Pós-graduação', 'Mestrado', 'Doutorado'], n,
                                          p=[0.15, 0.30, 0.30, 0.15, 0.07, 0.03]),
        'categoria_produto': np.random.choice(['Eletrônicos', 'Moda', 'Casa e Decoração', 'Saúde e Beleza',
                                              'Esportes', 'Livros', 'Alimentos', 'Brinquedos',
                                              'Ferramentas', 'Automotivo'], n),
        'cidade': np.random.choice(['São Paulo', 'Rio de Janeiro', 'Belo Horizonte', 'Salvador', 'Brasília',
                                   'Recife', 'Fortaleza', 'Curitiba', 'Manaus', 'Porto Alegre', 'Campinas',
                                   'Goiânia', 'Belém', 'Florianópolis', 'Vitória'], n),
        'meio_pagamento': np.random.choice(['Cartão de Crédito', 'Boleto', 'Pix',
                                           'Transferência Bancária', 'Cartão de Débito'], n,
                                          p=[0.40, 0.15, 0.30, 0.05, 0.10]),
        'faixa_renda': np.random.choice(['Até R$2.000', 'R$2.001 a R$4.000',
                                        'R$4.001 a R$8.000', 'Acima de R$8.000'], n,
                                       p=[0.25, 0.35, 0.25, 0.15]),
        'canal_aquisicao': np.random.choice(['Busca Orgânica', 'Redes Sociais', 'Email Marketing',
                                            'Indicação', 'Anúncios'], n,
                                           p=[0.25, 0.30, 0.15, 0.15, 0.15]),
        'idade': np.random.beta(2, 2.5, n) * 60 + 18,
        'valor_compra': np.random.lognormal(5.5, 0.9, n),
        'tempo_site_minutos': np.random.gamma(2, 5, n),
        'compras_anteriores': np.random.negative_binomial(3, 0.5, n),
        'itens_carrinho': np.random.poisson(3, n) + 1,
    })

    # Criando variável alvo
    probabilidade_compra = (
        0.3 * (df['compras_anteriores'] > 2).astype(int) +
        0.2 * (df['valor_compra'] > df['valor_compra'].median()).astype(int) +
        0.2 * (df['tempo_site_minutos'] > 10).astype(int) +
        0.15 * (df['meio_pagamento'] == 'Cartão de Crédito').astype(int) +
        0.15 * np.random.random(n)
    )
    df['nova_compra'] = (probabilidade_compra > 0.5).astype(int)

    # Arredondando valores
    df['valor_compra'] = df['valor_compra'].round(2)
    df['tempo_site_minutos'] = df['tempo_site_minutos'].round(2)
    df['idade'] = df['idade'].round(0)

    print("Dados gerados com sucesso!")
else:
    print("Dados já carregados da Parte 1!")

## 1. Ordinal Encoding

O **Ordinal Encoding** é uma técnica de codificação que preserva a ordem natural das categorias, atribuindo valores inteiros sequenciais de acordo com uma hierarquia definida.

### Como funciona:
- Define-se explicitamente a ordem das categorias
- Atribui valores inteiros crescentes (0, 1, 2, ...) respeitando essa ordem
- Diferente do Label Encoding que ordena alfabeticamente

### Quando usar:
- **Variáveis ordinais**: educação, faixas etárias, tamanhos (P/M/G)
- **Variáveis com hierarquia clara**: níveis de satisfação, rankings
- **Algoritmos sensíveis à ordem**: regressão linear, redes neurais

### Vantagens:
- ✅ Preserva relações de ordem importantes
- ✅ Mantém dimensionalidade baixa (1 coluna)
- ✅ Captura informação semântica da hierarquia

### Cuidados:
- ⚠️ Assume distâncias iguais entre categorias
- ⚠️ Requer conhecimento do domínio para definir ordem

In [ ]:
# @title
# Implementando Ordinal Encoding com mapeamento customizado
from sklearn.preprocessing import OrdinalEncoder

# Criando uma cópia do DataFrame
df_ordinal = df.copy()

# Definindo mapeamentos ordinais para variáveis com ordem natural
ordinal_mappings = {
    'nivel_educacao': [
        'Ensino Fundamental', 'Ensino Médio', 'Graduação',
        'Pós-graduação', 'Mestrado', 'Doutorado'
    ],
    'faixa_renda': [
        'Até R$2.000', 'R$2.001 a R$4.000', 'R$4.001 a R$8.000', 'Acima de R$8.000'
    ]
}

# Aplicando OrdinalEncoder com as categorias ordenadas
for col, categories in ordinal_mappings.items():
    encoder = OrdinalEncoder(categories=[categories], handle_unknown='use_encoded_value', unknown_value=-1)
    df_ordinal[f"{col}_ordinal"] = encoder.fit_transform(df_ordinal[[col]])

    # Mostrando o mapeamento
    print(f"\nMapeamento ordinal para '{col}':")
    for i, category in enumerate(categories):
        print(f"  {category} → {i}")

In [ ]:
# @title
# Comparando Ordinal Encoding com Label Encoding
for col in ordinal_mappings.keys():
    le = LabelEncoder()
    df_ordinal[f"{col}_label"] = le.fit_transform(df_ordinal[col])

    # Visualizando algumas linhas
    print(f"\nComparação para '{col}':")
    comparison = df_ordinal[[col, f"{col}_ordinal", f"{col}_label"]].head(10)
    print(comparison)

In [ ]:
# @title Visualizando o impacto da ordenação correta
# Análise para ver como a codificação correta pode revelar padrões
# Vamos verificar se há relação entre nível educacional e valor de compra

plt.figure(figsize=(12, 5))

# Com Label Encoding (ordem alfabética)
plt.subplot(1, 2, 1)
sns.boxplot(x=df_ordinal['nivel_educacao_label'], y=df_ordinal['valor_compra'])
plt.title('Valor de Compra vs. Nível Educacional\n(Label Encoding - Ordem Alfabética)')
plt.xlabel('Nível Educacional Codificado')
plt.ylabel('Valor de Compra (R$)')

# Com Ordinal Encoding (ordem natural)
plt.subplot(1, 2, 2)
sns.boxplot(x=df_ordinal['nivel_educacao_ordinal'], y=df_ordinal['valor_compra'])
plt.title('Valor de Compra vs. Nível Educacional\n(Ordinal Encoding - Ordem Natural)')
plt.xlabel('Nível Educacional Codificado')
plt.ylabel('Valor de Compra (R$)')

plt.tight_layout()
plt.show()

## 2. Target Encoding e seus Riscos

O **Target Encoding** (ou Mean Encoding) substitui cada categoria pela média da variável alvo para aquela categoria. É uma técnica poderosa que captura a relação direta entre categorias e a variável que queremos prever.

**Funcionamento:**
- Para cada categoria, calcula a média da variável alvo naquela categoria
- Substitui a categoria por esse valor médio
- No caso de variáveis alvo binárias (0/1), isso representa a proporção ou probabilidade da classe positiva

### ⚠️ **RISCOS DO TARGET ENCODING:**

**1. Overfitting (Sobreajuste)**
- Categorias com poucos exemplos podem ter médias não representativas
- O modelo pode memorizar as médias ao invés de aprender padrões

**2. Data Leakage (Vazamento de Dados)**
- Se calcular a média usando todo o dataset, informação do conjunto de teste vaza para o treino
- Sempre deve ser aplicado APENAS com dados de treino

**3. Categorias Raras**
- Categorias com 1-2 exemplos terão médias extremas (0 ou 1 para classificação binária)
- Podem causar predições incorretas e instáveis

### **TÉCNICAS DE MITIGAÇÃO:**

**1. Smoothing (Suavização)**
- Aproxima a média das categorias pequenas da média global
- Fórmula: `encoded = (n * mean_category + m * mean_global) / (n + m)`
- Onde n = contagem da categoria, m = parâmetro de smoothing

**2. Cross-Validation Encoding**
- Usa validação cruzada para calcular as médias
- Evita usar a própria observação no cálculo da média

**3. Leave-One-Out Encoding**
- Para cada registro, calcula a média excluindo ele mesmo
- Reduz overfitting mas aumenta o tempo de processamento

**4. Agrupamento de Categorias Raras**
- Agrupa categorias com poucas ocorrências antes de aplicar Target Encoding

Vamos implementar o Target Encoding com técnicas de mitigação:

In [ ]:
# @title Aplicando Target Encoding com smoothing para regularização
from category_encoders import TargetEncoder

# Criando uma cópia do DataFrame
df_target = df.copy()

# Variável alvo
target = 'nova_compra'

# Colunas para aplicar Target Encoding
cat_cols = ['categoria_produto', 'cidade', 'meio_pagamento', 'canal_aquisicao']

# Dividir dados em treino e teste para evitar data leakage
X_train, X_test, y_train, y_test = train_test_split(
    df_target[cat_cols],
    df_target[target],
    test_size=0.3,
    random_state=42
)

# Aplicando Target Encoding com smoothing para regularização
target_encoder = TargetEncoder(cols=cat_cols, smoothing=10)
X_train_encoded = target_encoder.fit_transform(X_train, y_train)
X_test_encoded = target_encoder.transform(X_test)

In [ ]:
# @title
# Visualizando o resultado do Target Encoding
# Visualizando o resultado do Target Encoding
print("Resultado do Target Encoding (treino):")
X_train_encoded.head()

In [ ]:
# @title Análise da taxa média de nova_compra por categoria

# Calculando manualmente a taxa média de nova_compra por categoria
category_stats = df_target.groupby('categoria_produto')['nova_compra'].agg(['mean', 'count']).reset_index()
category_stats.columns = ['categoria_produto', 'taxa_nova_compra', 'contagem']
category_stats = category_stats.sort_values('taxa_nova_compra', ascending=False)

print("Taxa média de nova_compra por categoria de produto:")
print(category_stats)

# Visualizando graficamente
plt.figure(figsize=(12, 5))
ax = sns.barplot(x='categoria_produto', y='taxa_nova_compra', data=category_stats)
plt.title('Taxa Média de Nova Compra por Categoria de Produto')
plt.xlabel('Categoria de Produto')
plt.ylabel('Taxa de Nova Compra')
plt.xticks(rotation=45)

# Anotando as contagens
for i, p in enumerate(ax.patches):
    count = category_stats.iloc[i]['contagem']
    ax.annotate(f'n={count}', (p.get_x() + p.get_width()/2., p.get_height() + 0.01),
               ha='center', va='center', fontsize=10, color='black')

plt.tight_layout()
plt.show()

## 3. Hash Encoding

O **Hash Encoding** (ou Feature Hashing) é uma técnica que usa funções hash para mapear categorias para um número fixo de colunas numéricas.

### Como funciona:
- Aplica uma função hash em cada categoria
- Mapeia o resultado para um número fixo de colunas (n_components)
- Múltiplas categorias podem mapear para a mesma coluna (colisão)

### Quando usar:
- **Altíssima cardinalidade**: milhares ou milhões de categorias
- **Categorias desconhecidas**: novas categorias em produção
- **Memória limitada**: controle sobre dimensionalidade
- **Dados em streaming**: não precisa conhecer todas as categorias

### Vantagens:
- ✅ Dimensionalidade controlada e previsível
- ✅ Funciona com categorias nunca vistas
- ✅ Eficiente em memória e processamento
- ✅ Não precisa armazenar mapeamentos

### Desvantagens:
- ❌ Colisões podem causar perda de informação
- ❌ Difícil interpretação das features resultantes
- ❌ Não há garantia de reversibilidade

In [ ]:
# @title Implementando Hash Encoding com controle de dimensionalidade
# Visualizando os valores codificados
from category_encoders import HashingEncoder

# Criando uma cópia do DataFrame
df_hash = df.copy()

# Selecionando a coluna de alta cardinalidade
col_high_card = 'cidade'

# Aplicando Hash Encoding
# n_components controla o número de colunas de saída
hash_encoder = HashingEncoder(cols=[col_high_card], n_components=8)
df_hash_encoded = hash_encoder.fit_transform(df_hash[col_high_card])

# Visualizando o resultado
print(f"Hash Encoding para '{col_high_card}':")
print(f"Número de categorias únicas: {df_hash[col_high_card].nunique()}")
print(f"Número de colunas geradas: {df_hash_encoded.shape[1]}")
print("\nPrimeiras linhas codificadas:")
comparison_hash = pd.concat([df_hash[col_high_card].reset_index(drop=True), df_hash_encoded], axis=1)
comparison_hash.head(10)

In [ ]:
# @title Demonstrando capacidade de lidar com categorias novas
# Demonstrando a capacidade de lidar com categorias desconhecidas
# Criamos um pequeno DataFrame com algumas cidades novas
new_cities = pd.DataFrame({
    'cidade': ['Porto Velho', 'João Pessoa', 'Natal', 'São Paulo', 'Rio de Janeiro']
})

# Aplicamos o mesmo encoder às novas cidades
new_cities_encoded = hash_encoder.transform(new_cities)

print("Hash Encoding aplicado a novas cidades (incluindo algumas que não estavam nos dados de treino):")
pd.concat([new_cities, new_cities_encoded], axis=1).head()

In [ ]:
# @title Análise do trade-off entre dimensões e colisões
# Vamos verificar como diferentes números de colunas afetam o desempenho

# Função para estimar colisões
def estimate_hash_collisions(col, n_components):
    n_categories = df[col].nunique()
    # Probabilidade aproximada pelo princípio do aniversário
    collision_prob = 1 - np.exp(-n_categories*(n_categories-1)/(2*n_components))
    return collision_prob

# Testando diferentes números de componentes
components_range = [2, 4, 8, 16, 32, 64]
collision_probs = [estimate_hash_collisions(col_high_card, n) for n in components_range]

# Visualizando
plt.figure(figsize=(10, 5))
plt.plot(components_range, collision_probs, marker='o')
plt.title(f'Probabilidade Estimada de Colisões vs. Número de Componentes\n({df[col_high_card].nunique()} categorias únicas)')
plt.xlabel('Número de Componentes')
plt.ylabel('Probabilidade de Colisão')
plt.xticks(components_range)
plt.grid(alpha=0.3)
plt.show()

#### Análise do Hash Encoding

O Hash Encoding é uma técnica poderosa para lidar com variáveis categóricas de altíssima cardinalidade ou quando esperamos encontrar novas categorias durante a implementação do modelo.

**Observações importantes:**
- Podemos controlar o número de features de saída, independentemente do número de categorias únicas
- A técnica funciona bem com categorias que nunca foram vistas nos dados de treinamento
- Há um trade-off entre o número de dimensões e a probabilidade de colisões
- Quanto mais categorias únicas, mais dimensões são necessárias para evitar colisões significativas

**Quando usar Hash Encoding:**
- Conjuntos de dados com milhares ou milhões de categorias únicas
- Quando novas categorias podem aparecer durante o uso do modelo
- Quando a memória é limitada e não podemos armazenar todos os mapeamentos
- Em sistemas de processamento de texto ou dados de alta dimensionalidade

## 4. Frequency Encoding

O **Frequency Encoding** substitui cada categoria pela sua frequência (contagem ou percentual) no dataset.

### Como funciona:
- Calcula a frequência de cada categoria no conjunto de treino
- Substitui a categoria pelo valor de frequência
- Pode usar contagem absoluta ou percentual

### Quando usar:
- **Frequência é informativa**: categorias raras vs. comuns importam
- **Redução de dimensionalidade**: alternativa ao One-Hot
- **Algoritmos baseados em árvores**: funcionam bem com esta codificação

### Vantagens:
- ✅ Mantém apenas 1 coluna por variável
- ✅ Captura informação sobre popularidade/raridade
- ✅ Simples de implementar e interpretar

### Desvantagens:
- ❌ Categorias diferentes podem ter mesma frequência
- ❌ Sensível à distribuição dos dados de treino
- ❌ Pode não capturar relações semânticas

In [ ]:
# @title Implementando Frequency Encoding
# Criando uma cópia do DataFrame
df_freq = df.copy()

# Aplicando Frequency Encoding a várias colunas
cat_cols = ['categoria_produto', 'cidade', 'meio_pagamento', 'canal_aquisicao']

for col in cat_cols:
    # Cálculo da frequência (contagem)
    frequency_map = df_freq[col].value_counts().to_dict()
    df_freq[f'{col}_freq_count'] = df_freq[col].map(frequency_map)

    # Cálculo da frequência relativa (percentual)
    frequency_percent_map = (df_freq[col].value_counts(normalize=True) * 100).to_dict()
    df_freq[f'{col}_freq_percent'] = df_freq[col].map(frequency_percent_map)

# Visualizando o resultado para uma coluna
col_example = 'cidade'

# Obtendo as frequências únicas
frequencies = df_freq.groupby(col_example)[[f'{col_example}_freq_count', f'{col_example}_freq_percent']].first()
frequencies = frequencies.sort_values(f'{col_example}_freq_count', ascending=False).reset_index()

# Calculando os bins para histograma com margens de segurança
count_values = frequencies[f'{col_example}_freq_count'].values
count_min = count_values.min() - 0.01
count_max = count_values.max() + 0.01
count_bins = np.linspace(count_min, count_max, 16)

percent_values = frequencies[f'{col_example}_freq_percent'].values
percent_min = percent_values.min() - 0.01
percent_max = percent_values.max() + 0.01
percent_bins = np.linspace(percent_min, percent_max, 16)

# Adicionando informações dos bins
frequencies['bin_contagem'] = pd.cut(frequencies[f'{col_example}_freq_count'],
                                    bins=count_bins,
                                    labels=range(1, 16))

frequencies['bin_percentual'] = pd.cut(frequencies[f'{col_example}_freq_percent'],
                                      bins=percent_bins,
                                      labels=range(1, 16))

# Tratando valores NaN nos bins
frequencies['bin_contagem'] = frequencies['bin_contagem'].fillna(1)
frequencies['bin_percentual'] = frequencies['bin_percentual'].fillna(1)

# Renomeando as colunas para melhor visualização
frequencies = frequencies.rename(columns={
    col_example: 'cidade',
    f'{col_example}_freq_count': 'freq_count',
    f'{col_example}_freq_percent': 'freq_percent'
})

# Formatando os percentuais para melhor visualização
frequencies['freq_percent'] = frequencies['freq_percent'].round(1)

# Convertendo bins para inteiro para melhor exibição
frequencies['bin_contagem'] = frequencies['bin_contagem'].astype(int)
frequencies['bin_percentual'] = frequencies['bin_percentual'].astype(int)

# Imprimindo o resultado formatado
print(f"Frequency Encoding para '{col_example}':")
print(frequencies.to_string(index=False))

In [ ]:
# @title Análise da relação entre frequência e variável alvo
# Verificando se a frequência está relacionada com a variável alvo
# Vamos analisar se categoria_produto mais comuns têm taxas diferentes de nova_compra

col = 'categoria_produto'
analysis = df_freq.groupby(col)[[f'{col}_freq_percent', 'nova_compra']].agg({
    f'{col}_freq_percent': 'first',
    'nova_compra': 'mean'
}).reset_index()
analysis.columns = [col, 'frequencia_percentual', 'taxa_nova_compra']
analysis = analysis.sort_values('frequencia_percentual', ascending=False)

# Visualizando
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
sns.barplot(x=col, y='frequencia_percentual', data=analysis)
plt.title(f'Frequência das Categorias\n({col})')
plt.xticks(rotation=80)
plt.ylabel('Frequência (%)')

plt.subplot(1, 2, 2)
sns.barplot(x=col, y='taxa_nova_compra', data=analysis)
plt.title(f'Taxa de Nova Compra por Categoria\n({col})')
plt.xticks(rotation=80)
plt.ylabel('Taxa de Nova Compra')

plt.tight_layout()
plt.show()

In [ ]:
# @title Correlação entre frequência e taxa de nova compra
# Correlação entre frequência e taxa de nova compra
correlation = analysis[['frequencia_percentual', 'taxa_nova_compra']].corr().iloc[0, 1]
print(f"Correlação entre frequência percentual e taxa de nova compra: {correlation:.4f}")

# Visualizando a relação
plt.figure(figsize=(10, 6))
sns.scatterplot(x='frequencia_percentual', y='taxa_nova_compra', data=analysis, s=100)

# Anotando as categorias
for i, row in analysis.iterrows():
    plt.annotate(row[col], (row['frequencia_percentual'], row['taxa_nova_compra']),
                 xytext=(5, 5), textcoords='offset points')

plt.title(f'Relação entre Frequência da Categoria e Taxa de Nova Compra\n({col})')
plt.xlabel('Frequência Percentual (%)')
plt.ylabel('Taxa de Nova Compra')
plt.grid(alpha=0.3)
plt.show()

#### Análise do Frequency Encoding

O Frequency Encoding é uma técnica simples mas eficaz para extrair informações sobre a importância ou raridade das categorias.

**Observações importantes:**
- Categorias mais frequentes recebem valores mais altos, categorias raras recebem valores mais baixos
- A distribuição de frequências depende muito da natureza dos dados
- Em alguns domínios, a frequência de uma categoria pode ser altamente preditiva
- A correlação entre frequência e variável alvo varia para diferentes variáveis categóricas

**Quando usar Frequency Encoding:**
- Quando a popularidade ou raridade de uma categoria é informativa
- Como uma técnica complementar a outros métodos de codificação
- Para algoritmos lineares e baseados em distância
- Quando queremos preservar a dimensionalidade original dos dados

## 5. Comparação Dimensional e Prática das Técnicas

Nesta seção, vamos comparar as técnicas de codificação em termos de:
- Impacto dimensional (número de colunas geradas)
- Uso de memória
- Preservação de informação
- Performance em modelos de Machine Learning

**Nota:** Usaremos modelos simples (Regressão Logística e Random Forest) como "caixas pretas" para demonstrar como diferentes codificações impactam o desempenho. O foco não é nos modelos em si (que serão estudados em detalhes a partir da Aula 11), mas sim em como as escolhas de codificação afetam os resultados.

In [ ]:
# @title Análise de dimensionalidade e uso de memória
# Vamos primeiro analisar o impacto dimensional e uso de memória de cada técnica

import sys

# Função para calcular o uso de memória de um DataFrame
def get_memory_usage(df):
    """Calcula o uso de memória em MB."""
    memory_bytes = df.memory_usage(deep=True).sum()
    memory_mb = memory_bytes / 1024 / 1024
    return memory_mb

# Função para analisar preservação de informação
def analyze_information_preservation(original_col, encoded_cols):
    """Analisa quanto da informação original foi preservada."""
    n_unique_original = original_col.nunique()

    # Para codificações de coluna única
    if isinstance(encoded_cols, pd.Series):
        n_unique_encoded = encoded_cols.nunique()
        preservation_ratio = n_unique_encoded / n_unique_original
        return preservation_ratio

    # Para codificações de múltiplas colunas (One-Hot, Binary)
    elif isinstance(encoded_cols, pd.DataFrame):
        # Calcula o número de combinações únicas possíveis
        unique_combinations = encoded_cols.drop_duplicates().shape[0]
        preservation_ratio = min(1.0, unique_combinations / n_unique_original)
        return preservation_ratio

    return 0

# Comparando diferentes técnicas para uma variável categórica
cat_col_example = 'categoria_produto'
df_comparison = df.copy()

# Dicionário para armazenar resultados
comparison_results = {}

# 1. Label Encoding
le = LabelEncoder()
label_encoded = pd.Series(le.fit_transform(df_comparison[cat_col_example]))
memory_label = get_memory_usage(pd.DataFrame({cat_col_example: label_encoded}))
info_preservation_label = analyze_information_preservation(df_comparison[cat_col_example], label_encoded)
comparison_results['Label'] = {
    'n_columns': 1,
    'memory_mb': memory_label,
    'info_preservation': info_preservation_label
}

# 2. One-Hot Encoding
ohe = OneHotEncoder(sparse_output=False, drop='first')
onehot_encoded = pd.DataFrame(ohe.fit_transform(df_comparison[[cat_col_example]]))
memory_onehot = get_memory_usage(onehot_encoded)
info_preservation_onehot = analyze_information_preservation(df_comparison[cat_col_example], onehot_encoded)
comparison_results['One-Hot'] = {
    'n_columns': onehot_encoded.shape[1],
    'memory_mb': memory_onehot,
    'info_preservation': info_preservation_onehot
}

# 3. Binary Encoding
be = BinaryEncoder(cols=[cat_col_example])
binary_encoded = be.fit_transform(df_comparison[[cat_col_example]])
memory_binary = get_memory_usage(binary_encoded)
info_preservation_binary = analyze_information_preservation(df_comparison[cat_col_example], binary_encoded)
comparison_results['Binary'] = {
    'n_columns': binary_encoded.shape[1] - 1,  # Subtraindo a coluna index
    'memory_mb': memory_binary,
    'info_preservation': info_preservation_binary
}

# 4. Target Encoding
te = TargetEncoder(cols=[cat_col_example], smoothing=10)
target_encoded = te.fit_transform(df_comparison[[cat_col_example]], df_comparison['nova_compra'])
memory_target = get_memory_usage(target_encoded)
info_preservation_target = analyze_information_preservation(df_comparison[cat_col_example], target_encoded[cat_col_example])
comparison_results['Target'] = {
    'n_columns': 1,
    'memory_mb': memory_target,
    'info_preservation': info_preservation_target
}

# 5. Hash Encoding
he = HashingEncoder(cols=[cat_col_example], n_components=8)
hash_encoded = he.fit_transform(df_comparison[[cat_col_example]])
memory_hash = get_memory_usage(hash_encoded)
# Hash tem perda de informação devido a colisões
info_preservation_hash = min(1.0, 8 / df_comparison[cat_col_example].nunique())  # Aproximação
comparison_results['Hash'] = {
    'n_columns': 8,
    'memory_mb': memory_hash,
    'info_preservation': info_preservation_hash
}

# 6. Frequency Encoding
freq_map = df_comparison[cat_col_example].value_counts().to_dict()
freq_encoded = pd.Series(df_comparison[cat_col_example].map(freq_map))
memory_freq = get_memory_usage(pd.DataFrame({cat_col_example: freq_encoded}))
info_preservation_freq = analyze_information_preservation(df_comparison[cat_col_example], freq_encoded)
comparison_results['Frequency'] = {
    'n_columns': 1,
    'memory_mb': memory_freq,
    'info_preservation': info_preservation_freq
}

# Criando DataFrame com resultados
comparison_df = pd.DataFrame(comparison_results).T
comparison_df.index.name = 'Técnica'
comparison_df = comparison_df.reset_index()

print(f"Comparação de técnicas de codificação para '{cat_col_example}':")
print(f"Cardinalidade original: {df_comparison[cat_col_example].nunique()} categorias únicas\n")
print(comparison_df.to_string(index=False))

In [ ]:
# @title Visualização comparativa de dimensionalidade e memória
# Criando visualizações para comparar as técnicas

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Gráfico 1: Número de colunas
ax1 = axes[0]
techniques = comparison_df['Técnica']
n_cols = comparison_df['n_columns']
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#FFA07A', '#DDA0DD']
ax1.bar(techniques, n_cols, color=colors)
ax1.set_title('Impacto Dimensional\n(Número de Colunas Geradas)')
ax1.set_xlabel('Técnica de Codificação')
ax1.set_ylabel('Número de Colunas')
ax1.tick_params(axis='x', rotation=45)
for i, v in enumerate(n_cols):
    ax1.text(i, v + 0.1, str(int(v)), ha='center', va='bottom')

# Gráfico 2: Uso de memória
ax2 = axes[1]
memory = comparison_df['memory_mb']
ax2.bar(techniques, memory, color=colors)
ax2.set_title('Uso de Memória\n(em MB)')
ax2.set_xlabel('Técnica de Codificação')
ax2.set_ylabel('Memória (MB)')
ax2.tick_params(axis='x', rotation=45)
for i, v in enumerate(memory):
    ax2.text(i, v + 0.001, f'{v:.3f}', ha='center', va='bottom')

# Gráfico 3: Preservação de informação
ax3 = axes[2]
info_pres = comparison_df['info_preservation'] * 100
ax3.bar(techniques, info_pres, color=colors)
ax3.set_title('Preservação de Informação\n(% da informação original)')
ax3.set_xlabel('Técnica de Codificação')
ax3.set_ylabel('Preservação (%)')
ax3.tick_params(axis='x', rotation=45)
ax3.set_ylim(0, 105)
for i, v in enumerate(info_pres):
    ax3.text(i, v + 1, f'{v:.1f}%', ha='center', va='bottom')

plt.tight_layout()
plt.show()

print("\nAnálise dos resultados:")
print("- One-Hot e Binary geram múltiplas colunas, aumentando dimensionalidade")
print("- Target e Frequency reduzem categorias a valores únicos, perdendo alguma informação")
print("- Hash Encoding oferece controle sobre dimensionalidade mas com perda por colisões")
print("- Label Encoding preserva toda informação em 1 coluna mas cria ordem artificial")

In [ ]:
# @title Preparação de dados com diferentes técnicas de codificação
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline

# Definindo colunas categóricas e numéricas para o modelo
cat_cols = ['sexo', 'estado_civil', 'nivel_educacao', 'categoria_produto',
           'cidade', 'meio_pagamento', 'faixa_renda', 'canal_aquisicao']
num_cols = ['idade', 'valor_compra', 'tempo_site_minutos', 'compras_anteriores', 'itens_carrinho']
target = 'nova_compra'

# Função para preparar dados com diferentes codificações
def prepare_encoded_data(df, encoding_method):
    # Cópia para não modificar o original
    data = df.copy()

    # Preparando X e y
    X = data[cat_cols + num_cols]
    y = data[target]

    # Dividindo em treino e teste
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=42
    )

    # Aplicando a codificação selecionada
    if encoding_method == 'label':
        # Label Encoding
        for col in cat_cols:
            le = LabelEncoder()
            X_train[col] = le.fit_transform(X_train[col])
            X_test[col] = le.transform(X_test[col])

    elif encoding_method == 'onehot':
        # One-Hot Encoding
        ohe = OneHotEncoder(sparse_output=False, drop='first', handle_unknown='ignore')
        cat_train = ohe.fit_transform(X_train[cat_cols])
        cat_test = ohe.transform(X_test[cat_cols])

        # Convertendo para DataFrame
        cat_cols_ohe = []
        for i, col in enumerate(cat_cols):
            cats = ohe.categories_[i][1:]
            for cat in cats:
                cat_cols_ohe.append(f"{col}_{cat}")

        # Combinando com colunas numéricas
        cat_train_df = pd.DataFrame(cat_train, columns=cat_cols_ohe, index=X_train.index)
        cat_test_df = pd.DataFrame(cat_test, columns=cat_cols_ohe, index=X_test.index)

        X_train = pd.concat([cat_train_df, X_train[num_cols]], axis=1)
        X_test = pd.concat([cat_test_df, X_test[num_cols]], axis=1)

    elif encoding_method == 'binary':
        # Binary Encoding
        be = BinaryEncoder(cols=cat_cols)
        X_train = be.fit_transform(X_train)
        X_test = be.transform(X_test)

    elif encoding_method == 'target':
        # Target Encoding
        te = TargetEncoder(cols=cat_cols, smoothing=10)
        X_train = te.fit_transform(X_train, y_train)
        X_test = te.transform(X_test)

    elif encoding_method == 'frequency':
        # Frequency Encoding (implementação manual)
        for col in cat_cols:
            # Calculando as frequências apenas nos dados de treino
            freq_map = X_train[col].value_counts(normalize=True).to_dict()
            # Aplicando aos dados de treino e teste
            X_train[col] = X_train[col].map(freq_map)
            # Para categorias novas no teste, usamos 0
            X_test[col] = X_test[col].map(freq_map).fillna(0)

    return X_train, X_test, y_train, y_test

In [ ]:
# @title
# Função para avaliar modelos de ML
# Função para avaliar modelos
def evaluate_models(X_train, X_test, y_train, y_test):
    results = {}

    # Modelo 1: Regressão Logística
    log_reg = LogisticRegression(max_iter=1000, random_state=42)
    log_reg.fit(X_train, y_train)
    log_reg_train_acc = log_reg.score(X_train, y_train)
    log_reg_test_acc = log_reg.score(X_test, y_test)

    # Cross Validation para LogReg
    log_reg_cv = cross_val_score(log_reg, X_train, y_train, cv=5, scoring='accuracy')

    # Modelo 2: Random Forest
    rf = RandomForestClassifier(n_estimators=100, random_state=42)
    rf.fit(X_train, y_train)
    rf_train_acc = rf.score(X_train, y_train)
    rf_test_acc = rf.score(X_test, y_test)

    # Cross Validation para Random Forest
    rf_cv = cross_val_score(rf, X_train, y_train, cv=5, scoring='accuracy')

    # Armazenando resultados
    results['LogReg'] = {
        'train_acc': log_reg_train_acc,
        'test_acc': log_reg_test_acc,
        'cv_mean': log_reg_cv.mean(),
        'cv_std': log_reg_cv.std()
    }

    results['RandomForest'] = {
        'train_acc': rf_train_acc,
        'test_acc': rf_test_acc,
        'cv_mean': rf_cv.mean(),
        'cv_std': rf_cv.std()
    }

    return results

# Lista de métodos de codificação para avaliar
encoding_methods = ['label', 'onehot', 'binary', 'target', 'frequency']

# Dicionário para armazenar resultados
all_results = {}

# Avaliar cada método
for method in encoding_methods:
    print(f"Avaliando codificação: {method}")
    X_train, X_test, y_train, y_test = prepare_encoded_data(df, method)
    all_results[method] = evaluate_models(X_train, X_test, y_train, y_test)

In [ ]:
# @title Compilando resultados da avaliação
# Compilando os resultados
results_df = pd.DataFrame(columns=["Codificação", "Modelo", "Treino Acc", "Teste Acc", "CV Mean", "CV Std"])

row_idx = 0
for encoding, models in all_results.items():
    for model, metrics in models.items():
        results_df.loc[row_idx] = [
            encoding,
            model,
            metrics['train_acc'],
            metrics['test_acc'],
            metrics['cv_mean'],
            metrics['cv_std']
        ]
        row_idx += 1

# Exibindo resultados organizados
print("\nResultados da Avaliação:")
results_df

####Definições resumidas para as colunas de avaliação:####

**Treino Acc**: Acurácia no conjunto de treinamento - mede o quanto o modelo consegue se ajustar aos dados que foram usados para treiná-lo.

**Teste Acc**: Acurácia no conjunto de teste - avalia o desempenho do modelo em dados novos, não utilizados durante o treinamento.

**CV Mean**: Média da validação cruzada - representa a acurácia média obtida quando o modelo é avaliado em múltiplas divisões diferentes dos dados.

**CV Std**: Desvio padrão da validação cruzada - indica a variabilidade do desempenho do modelo entre as diferentes divisões dos dados; valores menores sugerem maior estabilidade.

In [ ]:
# @title Visualização comparativa de performance por modelo
# Visualizando os resultados
plt.figure(figsize=(15, 10))

# Subplot para Regressão Logística
plt.subplot(2, 1, 1)
logreg_results = results_df[results_df['Modelo'] == 'LogReg']
x = np.arange(len(logreg_results))
width = 0.35

plt.bar(x - width/2, logreg_results['Treino Acc'], width, label='Treino')
plt.bar(x + width/2, logreg_results['Teste Acc'], width, label='Teste')

plt.xlabel('Método de Codificação')
plt.ylabel('Acurácia')
plt.title('Desempenho da Regressão Logística por Método de Codificação')
plt.xticks(x, logreg_results['Codificação'])
plt.legend()
plt.ylim(0.5, 1.0)
plt.grid(axis='y', alpha=0.3)

# Subplot para Random Forest
plt.subplot(2, 1, 2)
rf_results = results_df[results_df['Modelo'] == 'RandomForest']
x = np.arange(len(rf_results))

plt.bar(x - width/2, rf_results['Treino Acc'], width, label='Treino')
plt.bar(x + width/2, rf_results['Teste Acc'], width, label='Teste')

plt.xlabel('Método de Codificação')
plt.ylabel('Acurácia')
plt.title('Desempenho do Random Forest por Método de Codificação')
plt.xticks(x, rf_results['Codificação'])
plt.legend()
plt.ylim(0.5, 1.0)
plt.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# @title
# Comparação de acurácia entre modelos e técnicas
# Visualizando a diferença de desempenho entre os modelos
plt.figure(figsize=(12, 6))

# Preparando os dados
test_acc_pivot = results_df.pivot(index='Codificação', columns='Modelo', values='Teste Acc')

# Plotando
test_acc_pivot.plot(kind='bar', figsize=(12, 6))
plt.title('Comparação da Acurácia de Teste entre Modelos por Método de Codificação')
plt.xlabel('Método de Codificação')
plt.ylabel('Acurácia de Teste')
plt.ylim(0.5, 1.0)
plt.grid(axis='y', alpha=0.3)
plt.legend(title='Modelo')
plt.tight_layout()
plt.show()

## 🎯 Micro-atividade: Encoding Battle (10 min)

**Objetivo**: Comparar diferentes técnicas de codificação em um cenário prático

**Cenário**: Você tem um dataset com 3 variáveis categóricas de diferentes características:
- `categoria_A`: Baixa cardinalidade (5 categorias), nominal
- `categoria_B`: Média cardinalidade (20 categorias), ordinal  
- `categoria_C`: Alta cardinalidade (100+ categorias), nominal

**Sua tarefa**:
1. Para cada variável, escolha a técnica de codificação mais apropriada e justifique
2. Implemente as codificações escolhidas
3. Treine um modelo simples (LogisticRegression) com cada combinação
4. Compare os resultados e identifique a melhor estratégia

**Critérios de avaliação**:
- Justificativa técnica para cada escolha
- Correta implementação das técnicas
- Análise dos resultados obtidos
- Identificação de trade-offs (dimensionalidade vs. performance)

**Dica**: Considere criar um pequeno quadro comparativo com:
- Técnica escolhida
- Número de features resultantes
- Acurácia obtida
- Tempo de processamento

---

## 6. Tratamento de Categorias de Alta Cardinalidade

In [ ]:
# @title Criando variável de altíssima cardinalidade
# Exemplo: simulando uma variável de altíssima cardinalidade
# Vamos criar uma variável 'id_produto' com milhares de valores únicos

np.random.seed(42)
n_products = 5000  # 5000 produtos únicos
all_products = [f"PROD-{i:05d}" for i in range(1, n_products+1)]

# Distribuição de vendas de produtos segue uma lei de potência (poucos produtos muito populares)
product_probs = np.random.power(0.3, n_products)
product_probs /= product_probs.sum()

# Adicionando ao DataFrame
df_highcard = df.copy()
df_highcard['id_produto'] = np.random.choice(all_products, size=len(df), p=product_probs)

# Verificando a cardinalidade
print(f"Cardinalidade da variável 'id_produto': {df_highcard['id_produto'].nunique()} categorias únicas")

# Visualizando a distribuição (top 20 produtos)
top_products = df_highcard['id_produto'].value_counts().head(20)
print(f"\nDistribuição dos 20 produtos mais comuns:")
print(top_products)

In [ ]:
# @title Técnica de agrupamento de categorias raras
# Técnica: Agrupando categorias raras em uma única categoria "Outros"
def group_rare_categories(df, column, threshold=5, top_n=None):
    """Agrupa categorias raras em uma única categoria 'Outros'.

    Args:
        df: DataFrame
        column: Nome da coluna para agrupar
        threshold: Mínimo de ocorrências para manter a categoria
        top_n: Manter apenas as top N categorias mais frequentes

    Returns:
        Series com categorias agrupadas
    """
    counts = df[column].value_counts()

    if top_n is not None:
        # Manter apenas as top N categorias
        keep_cats = counts.nlargest(top_n).index.tolist()
    else:
        # Manter categorias com pelo menos threshold ocorrências
        keep_cats = counts[counts >= threshold].index.tolist()

    # Aplicando a transformação
    return df[column].apply(lambda x: x if x in keep_cats else 'Outros')

# Aplicando a técnica para manter os top 50 produtos
df_highcard['id_produto_top50'] = group_rare_categories(df_highcard, 'id_produto', top_n=50)

# Verificando o resultado
print(f"Cardinalidade original: {df_highcard['id_produto'].nunique()} categorias")
print(f"Cardinalidade após agrupamento: {df_highcard['id_produto_top50'].nunique()} categorias")

# Contando quantos registros foram agrupados como 'Outros'
n_others = (df_highcard['id_produto_top50'] == 'Outros').sum()
pct_others = n_others / len(df_highcard) * 100
print(f"\nRegistros na categoria 'Outros': {n_others} ({pct_others:.2f}%)")

In [ ]:
# @title Preparação de dados para comparação
# Visualizando a distribuição de frequência
product_counts = df_highcard['id_produto'].value_counts()

plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.hist(product_counts, bins=50, log=True)
plt.title('Distribuição de Frequência (escala log)')
plt.xlabel('Número de Ocorrências')
plt.ylabel('Número de Produtos (log)')

plt.subplot(1, 2, 2)
plt.plot(sorted(product_counts, reverse=True))
plt.title('Produtos Ordenados por Popularidade')
plt.xlabel('Ranking do Produto')
plt.ylabel('Número de Ocorrências')
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

### 6.1 Estratégia 1: Agrupamento de Categorias Raras

O agrupamento de categorias raras é a estratégia mais simples e interpretável para lidar com alta cardinalidade.

**Implementação:**
1. Identificar categorias com baixa frequência (threshold)
2. Agrupar todas em uma categoria "Outros" ou "Raro"
3. Manter apenas as N categorias mais frequentes

**Vantagens:**
- Reduz drasticamente a cardinalidade
- Mantém interpretabilidade
- Previne overfitting em categorias raras

**Desvantagens:**
- Perda de informação das categorias agrupadas
- Threshold arbitrário

### 6.2 Estratégia 2: Codificação baseada em Similaridade de Categorias

Quando as categorias têm estrutura semântica (prefixos, sufixos, hierarquias), podemos extrair features mais gerais.

**Exemplos práticos:**
- IDs de produtos com prefixos de categoria (ELET-12345)
- CEPs agrupados por região
- Emails agrupados por domínio

**Implementação:**
1. Identificar padrões nas strings
2. Extrair componentes relevantes
3. Criar novas features baseadas nos padrões

**Vantagens:**
- Preserva informação semântica
- Reduz dimensionalidade inteligentemente
- Mantém interpretabilidade

In [ ]:
# @title Simulando categorias com padrões semânticos
# Neste exemplo, vamos criar IDs de produtos que têm prefixos de categoria
# Ex: ELET-12345 (Eletrônicos), LIVR-67890 (Livros), etc.

np.random.seed(42)
category_prefixes = {
    'ELET': 'Eletrônicos',
    'LIVR': 'Livros',
    'MODA': 'Moda',
    'ESPO': 'Esportes',
    'CASA': 'Casa',
    'BELT': 'Beleza',
    'BRIN': 'Brinquedos',
    'AUTO': 'Automotivo'
}

# Gerando IDs com prefixos de categoria
def generate_product_id(prefix):
    suffix = np.random.randint(10000, 99999)
    return f"{prefix}-{suffix}"

# Simulando dados
df_semantic = df.copy()
prefixes = list(category_prefixes.keys())
selected_prefixes = np.random.choice(prefixes, size=len(df_semantic), p=[0.25, 0.15, 0.2, 0.1, 0.1, 0.05, 0.05, 0.1])

product_ids = [generate_product_id(prefix) for prefix in selected_prefixes]
df_semantic['product_id'] = product_ids
df_semantic['product_category'] = [category_prefixes[id.split('-')[0]] for id in product_ids]

# Verificando a cardinalidade
print(f"Cardinalidade de product_id: {df_semantic['product_id'].nunique()} categorias únicas")
print(f"Cardinalidade de product_category: {df_semantic['product_category'].nunique()} categorias únicas")

In [ ]:
# @title Extraindo o prefixo da categoria do ID do produto
def extract_category_prefix(product_id):
    """Extrai o prefixo de categoria do ID do produto."""
    prefix = product_id.split('-')[0]
    return category_prefixes.get(prefix, 'Outro')

# Aplicando a extração
df_semantic['extracted_category'] = df_semantic['product_id'].apply(extract_category_prefix)

# Verificando a correspondência
match = (df_semantic['extracted_category'] == df_semantic['product_category']).mean() * 100
print(f"Correspondência entre categorias extraídas e originais: {match:.2f}%")

# Agora podemos usar a categoria extraída ao invés do ID completo
print(f"\nUso das categorias extraídas:")
print(f"  - Original: {df_semantic['product_id'].nunique()} categorias únicas")
print(f"  - Agrupado: {df_semantic['extracted_category'].nunique()} categorias únicas")

### 6.3 Estratégia 3: Abordagens Avançadas para Alta Cardinalidade

Para cenários complexos, combinamos múltiplas técnicas para equilibrar performance e interpretabilidade.

**Abordagem híbrida recomendada:**
1. **Hash Encoding** para capturar detalhes (muitas dimensões)
2. **Agrupamento** das top N categorias (interpretável)
3. **Target Encoding** com regularização (performance)

**Quando usar:**
- Datasets com milhões de categorias
- Necessidade de interpretabilidade + performance
- Sistemas em produção com novas categorias

In [ ]:
# @title Técnica: Combinação de Feature Hashing com Target Encoding
# Esta abordagem ajuda a balancear interpretabilidade e desempenho

# 1. Criar features usando Hash Encoding para capturar padrões detalhados
hash_encoder = HashingEncoder(cols=['id_produto'], n_components=20)
df_hash_advanced = hash_encoder.fit_transform(df_highcard[['id_produto']])

# 2. Complementar com agrupamento de categorias para interpretabilidade
df_highcard['id_produto_grupos'] = group_rare_categories(df_highcard, 'id_produto', top_n=10)

# 3. One-Hot Encoding nas categorias agrupadas
ohe = OneHotEncoder(sparse_output=False, drop='first')
grouped_encoded = ohe.fit_transform(df_highcard[['id_produto_grupos']])
grouped_cols = [f"grupo_{i}" for i in range(grouped_encoded.shape[1])]
df_grouped_encoded = pd.DataFrame(grouped_encoded, columns=grouped_cols)

# 4. Combinar as abordagens
final_features = pd.concat([df_hash_advanced.reset_index(drop=True),
                            df_grouped_encoded.reset_index(drop=True)],
                           axis=1)

print(f"Características finais combinadas:")
print(f"  - Número total de colunas: {final_features.shape[1]}")
print(f"  - Features de hashing: {df_hash_advanced.shape[1]}")
print(f"  - Features interpretáveis: {df_grouped_encoded.shape[1]}")
print(f"\nPrimeiras linhas do resultado:")
final_features.head()

#### Resumo de Estratégias para Alta Cardinalidade

Ao lidar com variáveis categóricas de alta cardinalidade, considere estas estratégias:

1. **Agrupamento de Categorias Raras**
   - Manter apenas as N categorias mais frequentes
   - Agrupar todas as outras em uma categoria 'Outros'
   - Reduz cardinalidade e mitiga o problema de categorias raras
   - Mais interpretável, mas pode perder informações úteis

2. **Codificação baseada em Similaridade**
   - Extrair informações de padrões nos valores categóricos
   - Agrupar por prefixos, sufixos ou outras características comuns
   - Útil quando há estrutura semântica nos valores (ex: IDs de produtos)
   - Preserva informação contextual importante

3. **Feature Hashing Avançado**
   - Combinar Hash Encoding com representações interpretáveis
   - Controlar a dimensionalidade enquanto preserva alguma interpretabilidade
   - Equilibra desempenho do modelo e compreensão das variáveis
   - Abordagem híbrida para requisitos variados

4. **Outras Abordagens Úteis**
   - **Target Encoding**: Eficaz para variáveis de alta cardinalidade em problemas supervisionados
   - **Embeddings**: Para relações semânticas complexas (como palavras ou produtos relacionados)
   - **Análise de Componentes Principais**: Após uma codificação inicial para reduzir dimensionalidade
   - **Esquemas híbridos**: Adaptados ao problema e domínio específicos

A escolha da estratégia depende do problema específico, da natureza dos dados e dos requisitos de interpretabilidade e desempenho.

## 7. Melhores Práticas e Guidelines

### Diretrizes para Escolha de Técnicas de Codificação

**1. Análise da Natureza da Variável:**
- **Nominal**: One-Hot, Binary ou Hash Encoding
- **Ordinal**: Ordinal Encoding respeitando a ordem natural
- **Alta cardinalidade**: Target, Hash ou Frequency Encoding

**2. Considerações sobre o Algoritmo:**
- **Árvores de decisão**: Toleram bem Label Encoding
- **Modelos lineares**: Preferem One-Hot Encoding para nominais
- **Redes neurais**: Beneficiam-se de embeddings para alta cardinalidade

**3. Trade-offs Importantes:**
- **Dimensionalidade vs. Informação**: One-Hot preserva informação mas aumenta dimensões
- **Interpretabilidade vs. Performance**: Target Encoding melhora performance mas reduz interpretabilidade
- **Generalização vs. Ajuste**: Técnicas complexas podem causar overfitting

**4. Validação e Teste:**
- Sempre aplicar codificação APÓS split treino/teste
- Usar validação cruzada para comparar técnicas
- Monitorar sinais de data leakage

**5. Documentação:**
- Registrar justificativas para escolhas
- Documentar tratamento de categorias raras/novas
- Manter rastreabilidade das transformações

## 8. Tabela Comparativa Final

### 📊 Resumo Completo das Técnicas de Codificação

| Técnica | Dimensões | Preserva Ordem | Novas Categorias | Interpretabilidade | Quando Usar |
|---------|-----------|----------------|------------------|-------------------|-------------|
| **Label Encoding** | 1 | Artificial | ❌ Erro | Alta | Árvores, variáveis ordinais |
| **Ordinal Encoding** | 1 | ✅ Natural | ❌ Erro | Alta | Variáveis com ordem clara |
| **One-Hot Encoding** | n-1 | N/A | ✅ Ignora | Alta | Nominais, baixa cardinalidade |
| **Binary Encoding** | log₂(n) | Não | ❌ Erro | Média | Média cardinalidade |
| **Target Encoding** | 1 | N/A | ⚠️ Média global | Média | Alta cardinalidade, supervisionado |
| **Hash Encoding** | Fixo | N/A | ✅ Funciona | Baixa | Altíssima cardinalidade |
| **Frequency Encoding** | 1 | N/A | ⚠️ Zero | Alta | Quando frequência importa |

### 🎯 Recomendações Práticas

**Para começar um projeto:**
1. Análise exploratória: identifique tipo e cardinalidade
2. Inicie com One-Hot para nominais de baixa cardinalidade
3. Use Ordinal para variáveis com ordem natural
4. Considere Target Encoding para alta cardinalidade

**Para produção:**
- Hash Encoding para estabilidade com novas categorias
- Pipelines robustos com tratamento de valores ausentes
- Documentação clara dos mapeamentos

### ⚠️ Armadilhas Comuns

1. **Label Encoding em nominais**: cria ordem artificial
2. **One-Hot sem drop_first**: multicolinearidade
3. **Target Encoding sem regularização**: overfitting
4. **Hash com poucas dimensões**: muitas colisões
5. **Frequency em teste**: usar frequências do treino

---
